# Phần 4: Xây dựng Model

Thử nghiệm từ đơn giản đến phức tạp, so sánh bằng **cross-validation**
(công bằng hơn 1 lần train/test split) rồi đánh giá chi tiết trên 1 tập
test giữ riêng:

1. `LinearRegression` — baseline
2. `RidgeCV` — regularization, tự chọn alpha
3. `DecisionTree` — 1 cây đơn (dễ hiểu, dễ overfit)
4. `RandomForest` — ensemble nhiều cây, ổn định hơn
5. `XGBoost` — gradient boosting, thường mạnh nhất với dữ liệu dạng bảng

> Logic chi tiết nằm trong `src/train.py` (hàm `get_models()`,
> `cross_validate_models()`, `evaluate_on_holdout()`). Notebook này gọi lại
> và diễn giải kết quả.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_processed_data
from train import get_models, cross_validate_models, evaluate_on_holdout
from sklearn.model_selection import train_test_split

df = load_processed_data()
X = df.drop(columns=['price'])
y = np.log1p(df['price'])
print('Shape:', X.shape)

## Bước 1: So sánh model bằng Cross-Validation

**Vì sao không chỉ dùng 1 lần train/test split?** Một lần split có thể
"may rủi" — tập test ngẫu nhiên dễ hoặc khó hơn bình thường, khiến ta đánh
giá sai lệch về model nào thực sự tốt hơn. K-fold CV chia dữ liệu thành 5
phần, lần lượt dùng mỗi phần làm test và lấy trung bình → kết quả đáng tin
cậy hơn nhiều, đồng thời cho biết luôn độ *ổn định* của model qua độ lệch
chuẩn (std) giữa các fold.

In [ ]:
cv_results = cross_validate_models(X, y, n_splits=5)
cv_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(cv_results['model'], cv_results['CV_RMSLE_mean'],
         xerr=cv_results['CV_RMSLE_std'], color='steelblue')
plt.xlabel('RMSLE (thấp hơn = tốt hơn)')
plt.title('So sánh model bằng 5-fold Cross-Validation')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/figures/cv_model_comparison.png', dpi=110)
plt.show()

## Bước 2: Đánh giá chi tiết trên tập test giữ riêng

CV cho biết model nào tốt hơn *nói chung*, nhưng để có con số cụ thể
(MAE, MdAPE bằng USD) và để chọn model cuối cùng lưu lại, ta train trên
80% dữ liệu và đánh giá trên 20% còn lại — tập test này **không** được
dùng để tinh chỉnh bất cứ thứ gì, chỉ dùng để đánh giá cuối cùng.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

holdout_results = evaluate_on_holdout(X_train, X_test, y_train, y_test)
holdout_results

## Nhận xét kết quả

| Model | CV RMSLE | Test RMSLE | Test R²(log) | Test MAE |
|---|---|---|---|---|
| LinearRegression | 0.302 | 0.308 | 0.676 | $159,585 |
| RidgeCV | 0.302 | 0.308 | 0.677 | $159,560 |
| DecisionTree | 0.358 | 0.362 | 0.553 | $150,143 |
| RandomForest | 0.313 | 0.315 | 0.661 | $125,640 |
| **XGBoost** | **0.296** | **0.296** | **0.701** | **$112,725** |

**Quan sát quan trọng:**

1. **CV RMSLE và Test RMSLE gần như trùng khớp** cho mọi model → kết quả
   đáng tin cậy, không phải do tập test "may mắn".
2. **DecisionTree tệ nhất** — đúng như dự đoán, 1 cây đơn dễ overfit/kém
   tổng quát hơn ensemble.
3. **RandomForest chỉ ngang Linear/Ridge**, chưa vượt trội — vì Random
   Forest đã được set `max_depth=12` khá nông để tránh overfit, có thể
   tinh chỉnh thêm ở Phần 6.
4. **XGBoost thắng ở mọi metric** — gradient boosting thường mạnh nhất
   với dữ liệu dạng bảng (tabular), đúng như lý thuyết. Model này đã
   được tự động lưu vào `models/best_model.pkl`.
5. `MdAPE` (sai số % trung vị) của Linear/Ridge và XGBoost đều quanh
   13%, nhưng XGBoost có `MAE` bằng USD thấp hơn hẳn → XGBoost đặc biệt
   tốt hơn ở các căn nhà có giá trị lớn (nơi sai số USD ảnh hưởng nhiều).

## Bước 3: Feature importance của model tốt nhất (XGBoost)

In [ ]:
import joblib
best_model = joblib.load('../models/best_model.pkl')

importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
importances.sort_values().plot(kind='barh')
plt.title('Top 15 Feature Importance (XGBoost)')
plt.tight_layout()
plt.show()